In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
import warnings


Найти целевые переменные G_total и КГФ

In [3]:
warnings.filterwarnings('ignore')
df = pd.read_excel('mass_очистка.xlsx', header=1, skiprows=[2])
target_total = df[['G_total', 'КГФ']]

print(target_total.head(20))

     G_total         КГФ
0   2.782623  311.909400
1   3.697781  288.600300
2   4.515073  248.790600
3   5.217673  223.559100
4   5.765092  215.148600
5   3.076542  241.130794
6        NaN  188.000000
7        NaN  178.000000
8        NaN  169.000000
9        NaN  162.000000
10       NaN  230.000000
11       NaN  201.000000
12       NaN  172.000000
13       NaN  157.000000
14       NaN  141.000000
15       NaN  172.000000
16  4.232370  385.415949
17  4.714503  295.502880
18  5.727870  255.681127
19  5.775314  233.960444


Найти признаки переменные

In [ ]:
df = pd.read_excel('mass_очистка.xlsx', sheet_name='VU')
df = df.iloc[3:].reset_index(drop=True)
column_names = [
    '№', 'дд.мм.гггг', 'Глубина_манометра', 'Dшт', 'Руст_ср', 'Рзаб_ср', 'Pлин_ср',
    'Руст_кон', 'Рзаб_кон', 'Рлин_кон', 'Туст', 'Тна_шлейфе', 'Тзаб', 'Tлин',
    'Дебит_газа_ст', 'Дебит_ст_конд', 'Дебит_воды_ст', 'Дебит_смеси', 'Дебит_газа_л',
    'Дебит_кон_нест', 'Дебит_воды_л', 'Нэф', 'Рпл_КВД', 'Рпл_расчет', 'Рпл_Карноухов',
    'Pсб_атм', 'Pсб_бар', 'Ro_g', 'Ro_c', 'Ro_w', 'Удельная_плотность_газа',
    'G_total', 'КГФ', 'КГФ_т_тыс'
]
df.columns = column_names

In [16]:
for col in df.columns:
    df[col] = pd.to_numeric( df[col], errors='coerce')
print(f"Размер датасета: {df.shape}")
print(f"колонки: {df.columns.tolist()}")

Размер датасета: (92, 34)
колонки: ['№', 'дд.мм.гггг', 'Глубина_манометра', 'Dшт', 'Руст_ср', 'Рзаб_ср', 'Pлин_ср', 'Руст_кон', 'Рзаб_кон', 'Рлин_кон', 'Туст', 'Тна_шлейфе', 'Тзаб', 'Tлин', 'Дебит_газа_ст', 'Дебит_ст_конд', 'Дебит_воды_ст', 'Дебит_смеси', 'Дебит_газа_л', 'Дебит_кон_нест', 'Дебит_воды_л', 'Нэф', 'Рпл_КВД', 'Рпл_расчет', 'Рпл_Карноухов', 'Pсб_атм', 'Pсб_бар', 'Ro_g', 'Ro_c', 'Ro_w', 'Удельная_плотность_газа', 'G_total', 'КГФ', 'КГФ_т_тыс']


In [ ]:
target_vars=['G_total', 'КГФ' ]
exclude_cols = ['№', 'дд.мм.гггг']
feature_cols = []
for col in df.columns :
    if col not in exclude_cols:
        feature_cols.append(col)

['Глубина_манометра', 'Dшт', 'Руст_ср', 'Рзаб_ср', 'Pлин_ср', 'Руст_кон', 'Рзаб_кон', 'Рлин_кон', 'Туст', 'Тна_шлейфе', 'Тзаб', 'Tлин', 'Дебит_газа_ст', 'Дебит_ст_конд', 'Дебит_воды_ст', 'Дебит_смеси', 'Дебит_газа_л', 'Дебит_кон_нест', 'Дебит_воды_л', 'Нэф', 'Рпл_КВД', 'Рпл_расчет', 'Рпл_Карноухов', 'Pсб_атм', 'Pсб_бар', 'Ro_g', 'Ro_c', 'Ro_w', 'Удельная_плотность_газа', 'КГФ_т_тыс']


In [17]:
df_clean = df.dropna(subset=target_vars, how='all').copy() # delete varables od the column target_vars

# Заполняем пропуски медианой / Llenamos valores faltantes con mediana
for col in df_clean.columns:
    if df_clean[col].dtype in ['float64', 'int64']:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print(f"Признаков для анализа: {len(feature_cols)}")
print(f"Записей после очистки: {len(df_clean)}")
print(f"Размер датасета: {df_clean.shape}")

Признаков для анализа: 30
Записей после очистки: 92
Размер датасета: (92, 34)


функция расчёта gain ratio - funcion del calculo Gain Ratio

In [ ]:
def calculate_gain_ratio (df, feature_cols, target_cols):
    importances = {}
    y=df[target_cols].values

#нормализация целевой переменной - Normalization target variable 
    y = (y - np.min(y)) / (np.max(y) - np.min(y) + 1e-10)

    for feature in feature_cols:
        #extracts the values of the feature as an array
        #reshape(-1,1) convert to column
        x = df[feature].values.reshape(-1,1)
        #mutual_info_regression function of scikit-learn - Measures the dependen between X and y
        #random_state=42 set the seed for reproducibility
        #[0] take the firts value
        try:
            ig = mutual_info_regression(x, y, random_state=42) [0]
            unique_vals = len(np.unique(x[~np.isnan(x)]))
        #calculate entropy of partition / add 1 to avoid log(0)
            split_info = np.log2(unique_vals + 1)

            #calculate the Gain Ratio
            if split_info > 0 and ig > 0:
                gain_ratio = ig / split_info
            else:
                gain_ratio = 0
            
            importances[feature] = gain_ratio

        except Exception as e:
            importances[feature] = 0
            print(f"Ошибка в признаке {feature}: {e}")

    return importances

print("Расчёт Gain Ratio для G_total...")
importance_g_total = calculate_gain_ratio(df_clean, feature_cols, 'G_total')
print("Расчёт Gain Ratio для КГФ...")
importance_kgf = calculate_gain_ratio(df_clean, feature_cols, 'КГФ')

 # Усреднение важности / Promediado de importancia

avg_importance = {}
for feature in feature_cols:
    avg_importance[feature,0] = (importance_g_total.get(feature,0) + importance_kgf.get(feature,0)) /2
    #Ordenamiento descendente
    sorted_importance = sorted(avg_importance.items(), key=lambda x: x[1], reverse = True)
    
    print("\n Top-10 import indicate")

    for i, (feat, imp) in enumerate(sorted_importance[:10], 1):
            print(f"{i}. {feat}: {imp:.6f}")
    

Расчёт Gain Ratio для G_total...
Расчёт Gain Ratio для КГФ...


KeyError: 'кгф'

NameError: name 'target_cols' is not defined